# Notebook zum Einfügen der Daten in die Graphdatenbank Neo4j

In [ ]:
!pip install markdown

In [ ]:
!pip install sentence_transformers

In [ ]:
!pip install neo4j

## Hilfsfunktionen

In [ ]:
import sys
from pathlib import Path

# Projektroot ermitteln 
notebook_path = Path.cwd()  
project_root = notebook_path.parent.parent  

# Projektroot zu sys.path hinzufügen
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
import re
from sentence_transformers import SentenceTransformer

from src.common.Neo4jHandler import Neo4jHandler
from src.common.study_programs import get_study_programs_information
import os
from dotenv import load_dotenv

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Funktion zur Normalisierung der Namen, Funktion entfernt die Titel in den Namen, sodass nur Vor- und Nachname bleiben
def normalize_name(name: str) -> str:
    # Entferne akademische Titel
    """cleaned = re.sub(r"\bProf\.?", "", name)
    cleaned = re.sub(r"\bDr\.?", "", cleaned)
    cleaned = re.sub(r"rer\. nat\.?", "", cleaned)
    cleaned = re.sub(r"Dr-Ing\.?", "", cleaned)"""
    name = name.replace("Dr.-", "Dr. ")

    cleaned = re.sub(r"\bProf\.?\b", "", name)
    cleaned = re.sub(r"\bDr\.?\b(?!-)", "", cleaned)         
    cleaned = re.sub(r"\bDr-Ing\.?\b", "", cleaned)
    cleaned = re.sub(r"\brer\.?\s*nat\.?\b", "", cleaned)
    cleaned = re.sub(r"\bRA\b", "", cleaned)
    cleaned = re.sub(r"\bRechtsanwalt\b", "", cleaned)


    # Entferne Leerzeichen
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned

# Markdown-Datei laden
def load_markdown_text(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()
    return text

# Segmentierung des Texts
def segment_text(text, doc_title):
    # Segmentierung nach Paragraph (§ mit Nummer) und Titel
    pattern = r'##\s*§\s*(\d+)\s+([^\n]+)\n+((?:(?!##\s*§\s*\d+).)*)'
    matches = re.findall(pattern, text, re.DOTALL)
    segments = []
    for number, title_part, content in matches:
        # Titel zusammensetzen
        full_title = f"§ {number} {title_part.strip()}"
        # Content bereinigen
        content = content.strip()
        segments.append({
            'title': doc_title + ": " + full_title, 
            'content': full_title + "\n\n" + content
        })
    return segments


# Embeddings erzeugen
def embed_segments(segments):
    #model = SentenceTransformer(model_name)
    for segment in segments:
        segment['embedding'] = model.encode(segment['content']).tolist()
    return segments

In [ ]:
current_dir = os.getcwd()

project_root = os.path.abspath(os.path.join(current_dir, '..', '..'))

graph_data_path = os.path.join(project_root, 'data', 'graph')

## Neo4j Konfigurationsdaten

In [ ]:
env_path = os.path.join(project_root, '.env')

load_dotenv(dotenv_path=env_path )

neo4j_uri = os.getenv('NEO4J_URI')
neo4j_user = os.getenv('NEO4J_USERNAME')
neo4j_password = os.getenv('NEO4J_PASSWORD')

## Zuordnung von Standort zu Fachbereich und Fachbereich zu Studiengang

In [ ]:
locations = {
    "Iserlohn": [
        "Fachbereich Informatik und Naturwissenschaften",
        "Fachbereich Maschinenbau"
    ],
    "Hagen": [
        "Fachbereich Technische Betriebswirtschaft",
        "Fachbereich Elektrotechnik und Informationstechnik"
    ],
    "Meschede": [
        "Fachbereich Ingenieur- und Wirtschaftswissenschaften"
    ],
    "Soest": [
        "Fachbereich Agrarwirtschaft",
        "Fachbereich Bildungs- und Gesellschaftswissenschaften",
        "Fachbereich Elektrische Energietechnik",
        "Fachbereich Maschinenbau-Automatisierungstechnik"
    ]
}

study_programs = {
    "Fachbereich Informatik und Naturwissenschaften": [
        "Angewandte Biologie B.Sc.", 
        "Angewandte Informatik B.Sc. (berufsbegleitendes Verbundstudium)", 
        "Angewandte Informatik M.Sc. (berufsbegleitendes Verbundstudium)",
        "Angewandte Künstliche Intelligenz M.Sc. (berufsbegleitendes Verbundstudium)",
        "Informatik B.Sc.",
        "Life Science Engineering M.Sc. (berufsbegleitendes Verbundstudium)"
    ],
    "Fachbereich Maschinenbau": [
        "Automotive B.Eng.",
        "Fertigungstechnik B.Eng.",
        "Integrierte Produktentwicklung M.Eng.",
        "Kunststofftechnik B.Eng.",
        "Maschinenbau B.Eng. (berufsbegleitendes Verbundstudium)",
        "Maschinenbau B.Eng. Iserlohn",
        "Maschinenbau M.Eng. (berufsbegleitendes Verbundstudium) Iserlohn",
        "Mechatronik B.Eng.",
        "Mechatronik B.Eng. (berufsbegleitender Verbundstudiengang)",
        "Produktentwicklung / Konstruktion B.Eng.",
        "Forschungsmaster: Angewandte Wissenschaft in Technik, Wirtschaft und Gesellschaft M.Sc."
    ],
    "Fachbereich Agrarwirtschaft": [
        "Agrarwirtschaft B.Sc.",
        "Agrarwirtschaft M.Sc.",
        "Data Science für Agrarwirtschaft B.Sc.",
        "Digitale Technologien M.Eng.",
        "Forschungsmaster: Angewandte Wissenschaft in Technik, Wirtschaft und Gesellschaft M.Sc.",
        "Nachhaltige Ernährungssysteme B.Sc.",
        "Ökologie und Nachhaltigkeitsmanagement B.Sc."
    ],
    "Fachbereich Bildungs- und Gesellschaftswissenschaften": [
        "Frühpädagogik B.A.",
        "Frühpädagogik B.A. (berufsbegleitendes Verbundstudium)",
        "Frühpädagogik M.A. (berufsbegleitendes Verbundstudium)",
        "Medienpädagogik M.A. (berufsbegleitendes Verbundstudium)"
    ],
    "Fachbereich Maschinenbau-Automatisierungstechnik": [
        "Design- und Projektmanagement B.Sc.",
        "Digitale Technologien B.Eng. (Vollzeit / Dual) (künftig: Angewandte Informatik)",
        "Digitale Technologien M.Eng.",
        "Forschungsmaster: Angewandte Wissenschaft in Technik, Wirtschaft und Gesellschaft M.Sc.",
        "Maschinenbau B.Eng. Soest (Vollzeit / Teilzeit / Dual)",
        "Technik- und Unternehmensmanagement M.Eng. (berufsbegleitender weiterbildender Master-Verbundstudiengang)",
        "Wirtschaftsingenieurwesen B.Eng. Soest (Vollzeit / Dual)",
        "Wirtschaftsingenieurwesen-Maschinenbau B.Eng. (berufsbegleitendes Verbundstudium)"
    ],
    "Fachbereich Elektrische Energietechnik": [
        "Advanced Engineering and Engineering Management M.Sc.",
        "Business Administration with Informatics B.A.",
        "Digitale Technologien B.Eng. (Vollzeit / Dual) (künftig: Angewandte Informatik)",
        "Digitale Technologien M.Eng.",
        "Elektrotechnik B.Eng. Soest (Vollzeit / Dual)",
        "International Management & Information Systems M.A.",
        "International Management and Information Systems - Online M.A. (Postgraduate part-time study course)",
        "Wirtschaftsingenieurwesen B.Eng. Soest (Vollzeit / Dual)"
    ],
    "Fachbereich Ingenieur- und Wirtschaftswissenschaften": [
        "Angewandte Betriebswirtschaftslehre B.A. (Vollzeit / Teilzeit)",
        "Data Science M.Sc. (berufsbegleitend)",
        "Elektrotechnik B.Eng. Meschede (Vollzeit / Teilzeit)",
        "Elektrotechnik M.Eng.",
        "Forschungsmaster: Angewandte Wissenschaft in Technik, Wirtschaft und Gesellschaft M.Sc.",
        "International Management B.A. (Vollzeit / Teilzeit)",
        "Management für Ingenieur- und Naturwissenschaften MBA (berufsbegleitender weiterbildender Master-Verbundstudiengang)",
        "Maschinenbau B.Eng. Meschede (Vollzeit / Teilzeit)",
        "Maschinenbau M.Eng. (berufsbegleitendes Verbundstudium) Meschede",
        "Nachhaltiges Tourismusmanagement B.A.",
        "Strategisches Management M.A.",
        "Wirtschaft B.A. (Vollzeit / Teilzeit)",
        "Wirtschaftsinformatik B.Sc. Meschede (Vollzeit / Teilzeit)",
        "Wirtschaftsingenieurwesen B.Eng. Meschede (Vollzeit / Teilzeit)",
        "Wirtschaftspsychologie B.Sc.",
        "Wirtschaftspsychologie M.Sc. mit Schwerpunkt Coaching & Change"
    ],
    "Fachbereich Technische Betriebswirtschaft": [
        "Betriebswirtschaftslehre B.Sc.",
        "Betriebswirtschaftslehre B.Sc. (berufsbegleitendes Verbundstudium)",
        "Informatics and Business M.Sc.",
        "International Business B.Sc.",
        "Management für Ingenieur- und Naturwissenschaften MBA (berufsbegleitender weiterbildender Master-Verbundstudiengang) Hagen",
        "Wirtschaftsinformatik B.Sc. Hagen",
        "Wirtschaftsingenieurwesen B.Sc. (berufsbegleitendes Verbundstudium)",
        "Wirtschaftsingenieurwesen B.Sc. Hagen",
        "Wirtschaftsingenieurwesen M.Sc.",
        "Wirtschaftsrecht LL.B. (berufsbegleitendes Verbundstudium)",
        "Wirtschaftsrecht LL.M. (berufsbegleitender weiterbildender Master-Verbundstudiengang)"
    ],
    "Fachbereich Elektrotechnik und Informationstechnik": [
        "Elektrotechnik B.Eng. (berufsbegleitendes Verbundstudium)",
        "Elektrotechnik B.Eng. Hagen",
        "Elektrotechnik M.Eng. (berufsbegleitendes Verbundstudium)",
        "Medieninformatik B.Sc.",
        "Medizintechnik B.Eng.",
        "Medizintechnik M.Eng.",
        "Robotik B.Eng.",
        "Technische Informatik B.Eng.",
        "Forschungsmaster: Angewandte Wissenschaft in Technik, Wirtschaft und Gesellschaft M.Sc."
    ]
}


## Zuordnung von Studiengang zur Prüfungsordnung als Markdown

In [ ]:
study_program_pruefungsordnung = {
    "Angewandte Biologie B.Sc.": ["pruefungsordnungenMd/Iserlohn/Informatik und Naturwissenschaften/Angewandte_Biologie_Bachelor.md"],
    "Angewandte Informatik B.Sc. (berufsbegleitendes Verbundstudium)": ["pruefungsordnungenMd/Iserlohn/Informatik und Naturwissenschaften/Angewandte_Informatik_Bachelor_Verbund.md"],
    "Angewandte Informatik M.Sc. (berufsbegleitendes Verbundstudium)": ["pruefungsordnungenMd/Iserlohn/Informatik und Naturwissenschaften/Angewandte_Informatik_Master_Verbund.md"],
    "Angewandte Künstliche Intelligenz M.Sc. (berufsbegleitendes Verbundstudium)": ["pruefungsordnungenMd/Iserlohn/Informatik und Naturwissenschaften/Angewandte_KI_Master_Verbund.md"],
    "Informatik B.Sc.": ["pruefungsordnungenMd/Iserlohn/Informatik und Naturwissenschaften/Informatik_Bachelor.md"],
    "Life Science Engineering M.Sc. (berufsbegleitendes Verbundstudium)": ["pruefungsordnungenMd/Iserlohn/Informatik und Naturwissenschaften/Life_Science_Engineering_Master_Verbund.md"],
    "Automotive B.Eng.": ["pruefungsordnungenMd/Iserlohn/Maschinenbau/Automotive_Bachelor.md", "pruefungsordnungenMd/Iserlohn/Maschinenbau/Automotive_Bachelor_Aenderung_1.md", "pruefungsordnungenMd/Iserlohn/Maschinenbau/Automotive_Bachelor_Aenderung_2.md"],
    "Fertigungstechnik B.Eng.": ["pruefungsordnungenMd/Iserlohn/Maschinenbau/Fertigungstechnik_Bachelor.md"],
    "Integrierte Produktentwicklung M.Eng.": ["pruefungsordnungenMd/Iserlohn/Maschinenbau/Integrierte_Produktentwicklung_Master.md"],
    "Kunststofftechnik B.Eng.": ["pruefungsordnungenMd/Iserlohn/Maschinenbau/Fertigungstechnik_Bachelor.md"],
    "Maschinenbau B.Eng. (berufsbegleitendes Verbundstudium)": ["pruefungsordnungenMd/Iserlohn/Maschinenbau/Maschinenbau_Bachelor_Verbund.md", "pruefungsordnungenMd/Iserlohn/Maschinenbau/Maschinenbau_Bachelor_Verbund_Aenderung_1.md", "pruefungsordnungenMd/Iserlohn/Maschinenbau/Maschinenbau_Bachelor_Verbund_Aenderung_2.md", "pruefungsordnungenMd/Iserlohn/Maschinenbau/Maschinenbau_Bachelor_Verbund_Aenderung_3.md"],
    "Maschinenbau B.Eng. Iserlohn": ["pruefungsordnungenMd/Iserlohn/Maschinenbau/Maschinenbau_Bachelor.md"],
    "Maschinenbau M.Eng. (berufsbegleitendes Verbundstudium) Iserlohn": ["pruefungsordnungenMd/Iserlohn/Maschinenbau/Maschinenbau_Master_Verbund.md"],
    "Mechatronik B.Eng.": ["pruefungsordnungenMd/Iserlohn/Maschinenbau/Mechatronik_Bachelor.md"],
    "Mechatronik B.Eng. (berufsbegleitender Verbundstudiengang)": ["pruefungsordnungenMd/Iserlohn/Maschinenbau/Mechatronik_Bachelor_Verbund.md"],
    "Produktentwicklung / Konstruktion B.Eng.": ["pruefungsordnungenMd/Iserlohn/Maschinenbau/Produktentwicklung-Konstruktion_Bachelor.md"],
    "Agrarwirtschaft B.Sc.": ["pruefungsordnungenMd/Soest/Agrarwirtschaft/Agrarwirtschaft_Bachelor.md"],
    "Agrarwirtschaft M.Sc.": ["pruefungsordnungenMd/Soest/Agrarwirtschaft/Agrarwirtschaft_Master.md"],
    "Data Science für Agrarwirtschaft B.Sc.": ["pruefungsordnungenMd/Soest/Agrarwirtschaft/Data_Science_Agrarwirtschaft_Bachelor.md"],
    "Nachhaltige Ernährungssysteme B.Sc.": ["pruefungsordnungenMd/Soest/Agrarwirtschaft/Nachhaltige_Ernaehrungssysteme_Bachelor.md"],
    "Ökologie und Nachhaltigkeitsmanagement B.Sc.": ["pruefungsordnungenMd/Soest/Agrarwirtschaft/Oekologie_und_Nachhaltigkeitsmanagement_Bachelor.md"],
    "Frühpädagogik B.A.": ["pruefungsordnungenMd/Soest/Bildungs- und Gesellschaftswissenschaften/Fruehpaedagogik_Bachelor.md"],
    "Frühpädagogik B.A. (berufsbegleitendes Verbundstudium)": ["pruefungsordnungenMd/Soest/Bildungs- und Gesellschaftswissenschaften/Fruehpaedagogik_Bachelor_Verbund.md"],
    "Frühpädagogik M.A. (berufsbegleitendes Verbundstudium)": ["pruefungsordnungenMd/Soest/Bildungs- und Gesellschaftswissenschaften/Fruehpaedagogik_Master_Verbund.md"],
    "Medienpädagogik M.A. (berufsbegleitendes Verbundstudium)": ["pruefungsordnungenMd/Soest/Bildungs- und Gesellschaftswissenschaften/Medienpaedagogik_Master_Verbund.md"],
    "Design- und Projektmanagement B.Sc.": ["pruefungsordnungenMd/Soest/Maschinenbau-Automatisierungstechnik/Design_und_Projektmanagement_Bachelor.md"],
    "Digitale Technologien B.Eng.": ["pruefungsordnungenMd/Soest/Elektrische Energietechnik/Digitale_Technologien_Bachlor.md"],
    "Digitale Technologien M.Eng.": ["pruefungsordnungenMd/Soest/Elektrische Energietechnik/Digitale_Technologien_Master.md"],
    "Maschinenbau B.Eng. Soest": ["pruefungsordnungenMd/Soest/Maschinenbau-Automatisierungstechnik/Maschinenbau_Bachelor.md"],
    "Technik- und Unternehmensmanagement M.Eng. (berufsbegleitender weiterbildender Master-Verbundstudiengang)": ["pruefungsordnungenMd/Soest/Maschinenbau-Automatisierungstechnik/Technik_und_Unternehmensmanagement_Master_Verbund.md"],
    "Elektrotechnik B.Eng. Meschede": ["pruefungsordnungenMd/Meschede/Ingenieur- und Wirtschaftswissenschaften/Elektrotechnik_Bachelor.md"],
    "Elektrotechnik M.Eng.": ["pruefungsordnungenMd/Meschede/Ingenieur- und Wirtschaftswissenschaften/Elektrotechnik_Master.md"],
    "Angewandte Betriebswirtschaftslehre B.A.": ["pruefungsordnungenMd/Meschede/Ingenieur- und Wirtschaftswissenschaften/Angewandte_BWL_Bachelor.md"],
    "Data Science M.Sc. (berufsbegleitend)": ["pruefungsordnungenMd/Meschede/Ingenieur- und Wirtschaftswissenschaften/Data_Science_Master_Verbund.md"],
    "International Management B.A.": ["pruefungsordnungenMd/Meschede/Ingenieur- und Wirtschaftswissenschaften/International_Management_Bachelor.md"],
    "Management für Ingenieur- und Naturwissenschaften MBA (berufsbegleitender weiterbildender Master-Verbundstudiengang)": ["pruefungsordnungenMd/Hagen/Technische Betriebswirtschaft/MBA_Master_Verbund.md"],
    "Maschinenbau B.Eng. Meschede": ["pruefungsordnungenMd/Meschede/Ingenieur- und Wirtschaftswissenschaften/Maschinenbau_Bachelor.md"],
    "Maschinenbau M.Eng. (berufsbegleitendes Verbundstudium)": ["pruefungsordnungenMd/Meschede/Ingenieur- und Wirtschaftswissenschaften/Maschinenbau_Master_Verbund.md"],
    "Strategisches Management M.A.": ["pruefungsordnungenMd/Meschede/Ingenieur- und Wirtschaftswissenschaften/Strategisches_Management_Master.md"],
    "Wirtschaft B.A.": ["pruefungsordnungenMd/Meschede/Ingenieur- und Wirtschaftswissenschaften/Wirtschaft_Bachelor.md"],
    "Wirtschaftsinformatik B.Sc. Meschede": ["pruefungsordnungenMd/Meschede/Ingenieur- und Wirtschaftswissenschaften/Wirtschaftsinformatik_Bachelor.md"],
    "Wirtschaftsingenieurwesen B.Eng. Meschede": ["pruefungsordnungenMd/Meschede/Ingenieur- und Wirtschaftswissenschaften/Wirtschaftsingenieurwesen_Bachelor.md"],
    "Wirtschaftspsychologie B.Sc.": ["pruefungsordnungenMd/Meschede/Ingenieur- und Wirtschaftswissenschaften/Wirtschaftspsychologie_Bachelor.md"],
    "Wirtschaftspsychologie M.Sc. mit Schwerpunkt Coaching & Change": ["pruefungsordnungenMd/Meschede/Ingenieur- und Wirtschaftswissenschaften/Wirtschaftspsychologie_Master.md"],
    "Betriebswirtschaftslehre B.Sc.": ["pruefungsordnungenMd/Hagen/Technische Betriebswirtschaft/Betriebswirtschaftslehre_Bachelor.md"],
    "Betriebswirtschaftslehre B.Sc. (berufsbegleitendes Verbundstudium)": ["pruefungsordnungenMd/Hagen/Technische Betriebswirtschaft/Betriebswirtschaftslehre_Bachelor_Verbund.md"],
    "Informatics and Business M.Sc.": ["pruefungsordnungenMd/Hagen/Technische Betriebswirtschaft/Informatics_and_Business_Master.md"],
    "International Business B.Sc.": ["pruefungsordnungenMd/Hagen/Technische Betriebswirtschaft/International_Business_Bachelor.md"],
    "Management für Ingenieur- und Naturwissenschaften MBA (berufsbegleitender weiterbildender Master-Verbundstudiengang) Hagen": ["pruefungsordnungenMd/Hagen/Technische Betriebswirtschaft/MBA_Master_Verbund.md"],
    "Wirtschaftsinformatik B.Sc. Hagen": ["pruefungsordnungenMd/Hagen/Technische Betriebswirtschaft/Wirtschaftsinformatik_Bachelor.md"],
    "Wirtschaftsingenieurwesen B.Sc. (berufsbegleitendes Verbundstudium)": ["pruefungsordnungenMd/Hagen/Technische Betriebswirtschaft/Wirtschaftsingenieurwesen_Bachelor_Verbund.md"],
    "Wirtschaftsingenieurwesen B.Sc. Hagen": ["pruefungsordnungenMd/Hagen/Technische Betriebswirtschaft/Wirtschaftsingenieurwesen_Bachelor.md"],
    "Wirtschaftsingenieurwesen M.Sc.": ["pruefungsordnungenMd/Hagen/Technische Betriebswirtschaft/Wirtschaftsingenieurwesen_Master.md"],
    "Wirtschaftsrecht LL.B. (berufsbegleitendes Verbundstudium)": ["pruefungsordnungenMd/Hagen/Technische Betriebswirtschaft/Wirtschaftsrecht_Bachelor.md"],
    "Wirtschaftsrecht LL.M. (berufsbegleitender weiterbildender Master-Verbundstudiengang)": ["pruefungsordnungenMd/Hagen/Technische Betriebswirtschaft/Wirtschaftsrecht_Master.md"],
    "Elektrotechnik B.Eng. (berufsbegleitendes Verbundstudium)": ["pruefungsordnungenMd/Hagen/Elektrotechnik und Informationstechnik/Elektrotechnik_Bachelor_Verbund.md"],
    "Elektrotechnik B.Eng. Hagen": ["pruefungsordnungenMd/Hagen/Elektrotechnik und Informationstechnik/Elektrotechnik_Bachelor.md"],
    "Elektrotechnik M.Eng. (berufsbegleitendes Verbundstudium)": ["pruefungsordnungenMd/Hagen/Elektrotechnik und Informationstechnik/Elektrotechnik_Master_Verbund.md"],
    "Medieninformatik B.Sc.": ["pruefungsordnungenMd/Hagen/Elektrotechnik und Informationstechnik/Medieninformatik_Bachelor.md"],
    "Medizintechnik B.Eng.": ["pruefungsordnungenMd/Hagen/Elektrotechnik und Informationstechnik/Medizintechnik_Bachelor.md"],
    "Robotik B.Eng.": ["pruefungsordnungenMd/Hagen/Elektrotechnik und Informationstechnik/Robotik_Bachelor.md"],
    "Technische Informatik B.Eng.": ["pruefungsordnungenMd/Hagen/Elektrotechnik und Informationstechnik/Technische_Informatik_Bachelor.md"],
}

## Daten in Neo4j einfügen

In [ ]:
neo_handler = Neo4jHandler(neo4j_uri, neo4j_user, neo4j_password)

# Standorte und Fachbereiche speichern
neo_handler.save_locations(locations)
# Studiengänge speichern
neo_handler.save_study_programs(study_programs)
# Studiengangsinformationen erweitern
neo_handler.update_studyprogram_info(get_study_programs_information())
# Dokument Knoten zu den Studiengängen erstellen
neo_handler.save_document_node_for_study_program(study_programs)
# Personen importieren
neo_handler.import_persons_from_csv(os.path.join(graph_data_path, "mitarbeiter.csv"))

neo_handler.close()

Im folgenden Code werden die Markdown Dateien der Prüfungsordnungen ausgelesen und in Paragraphen unterteilt. Die Paragraphen werden vektorisiert und für jeden Paragraph wird ein Segmentknoten in der Datenbank erstellt. Dieser Segmentknoten ist mit dem jeweiligen Dokumentknoten verbunden

In [ ]:
neo_handler = Neo4jHandler(neo4j_uri, neo4j_user, neo4j_password)


for study_program, paths in study_program_pruefungsordnung.items():
    for path in paths:
        if path:
            path = os.path.join(graph_data_path, path)
            title = "Prüfungsordnung " + study_program
            text = load_markdown_text(path)
            segments = segment_text(text, title)
            segments = embed_segments(segments)
            neo_handler.save_segments(segments, title)

title = "Rahmenprüfungsordnung"

text = load_markdown_text(os.path.join(graph_data_path, "pruefungsordnungenMd/RPO/RPO_2018_endgueltig.md"))
segments = segment_text(text, title)
segments = embed_segments(segments)
neo_handler.save_segments(segments, title)

neo_handler.close()

## Informationen aus den Modulhandbüchern speichern

Im folgenden werden die Informationen zu den Modulhandbüchern aus der pickle Datei geladen. Beispielhaft werden die Informationen des Moduls `Cloud Computing` aus dem Studiengang `Angewandte Informatik M.Sc. (berufsbegleitendes Verbundstudium)` asugegeben

In [ ]:
import pickle
import os

with open(os.path.join(graph_data_path, "Modulhandbuecher.pkl"), "rb") as f:
    fachbereich = pickle.load(f)
    
for key, value in fachbereich["Angewandte Informatik M.Sc. (berufsbegleitendes Verbundstudium)"]["Compilerbau"].items():
    print(key, value)

Mit dem folgenden Code werden die Modulknoten mit den geladenen Informationen erzeugt. Die Modulknoten werden jeweils mit dem Dokumentknoten zum Modulshandbuch des Studiengangs verbunden. Zudem werden die einzelnen Module mit der Lehrperson verbunden.

In [ ]:
neo_handler = Neo4jHandler(neo4j_uri, neo4j_user, neo4j_password)

for stud_name, moduls in fachbereich.items():
    neo_handler.save_modules_for_handbook(stud_name, moduls)
    
neo_handler.close()

## Abfragen auf den Graphdaten

In [ ]:
neo_handler = Neo4jHandler(neo4j_uri, neo4j_user, neo4j_password)

# Query vektorisieren
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
query_text = "Projektarbeit"
query_vector = model.encode(query_text).tolist()

# Fachbereiche am Standort ermitteln
result = neo_handler.find_departments_by_location("Iserlohn")

print(result)

# Studiengänge eines Fachbereichs ermitteln
result = neo_handler.find_studyprograms_by_department(result[0])

print(result)

# Standort zu einem Studiengang ermitteln
result = neo_handler.find_location_by_studyprogram("Informatik B.Sc.")

print(result)

# Information anhand des Query-Vektors für einen Studiengang suchen
result = neo_handler.find_studyprogram_segments_similarity("Angewandte Informatik M.Sc. (berufsbegleitendes Verbundstudium)", query_vector)

for res in result:
    print(res['segment_text'])

neo_handler.close()

In [ ]:
neo_handler = Neo4jHandler(neo4j_uri, neo4j_user, neo4j_password)

# Modulinformationen aus dem Graph auslesen
res = neo_handler.find_modul_info_by_studyprogram("Angewandte Informatik M.Sc. (berufsbegleitendes Verbundstudium)", "Compilerbau")

for key, value in res['modul'].items():
    print(key, value)

neo_handler.close()

## Code zum Löschen aller Daten

In [ ]:
neo_handler = Neo4jHandler(neo4j_uri, neo4j_user, neo4j_password)
neo_handler.delete_all_data()
neo_handler.close()